# Practical P20: Embeddings & Vector Stores with LangChain & Google Gemini
**Course**: PGDCA — Hands-On Large Language Models  
**Unit 3**: LLM Frameworks for Application Development  
**Syllabus Topic**: 3.1-3.2 LangChain framework: vector stores, embeddings, retrieval  
**Model Provider**: Google Gemini (`models/text-embedding-004` & `gemini-1.5-flash`) via `langchain-google-genai`  
**Learning Outcome**: Understand vector embeddings, compute cosine similarity metrics, utilize LangChain's `Document` abstraction, and perform semantic similarity search using real Google Gemini embeddings and vector stores in an LCEL RAG pipeline.

## Part 1: Theoretical Foundations — Vector Embeddings & Similarity

### 1.1 What are Embeddings?
Vector embeddings represent unstructured text as arrays of continuous numbers (vectors) in a high-dimensional mathematical space (e.g., 768 dimensions for Google Gemini `text-embedding-004`).
* Sentences with **similar meanings** are positioned close together in vector space.
* Unrelated sentences are positioned far apart.

$$\text{"LangChain framework"} \approx \text{"LLM orchestration tool"} \quad \not\approx \text{"Baking chocolate chip cookies"}$$

### 1.2 Mathematical Metric: Cosine Similarity
To measure how close two vectors $\mathbf{u}$ and $\mathbf{v}$ are, we compute the cosine of the angle between them:

$$\text{Cosine Similarity}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|} = \frac{\sum_{i=1}^n u_i v_i}{\sqrt{\sum_{i=1}^n u_i^2} \sqrt{\sum_{i=1}^n v_i^2}}$$

* **Score = 1.0**: Identical semantic direction.
* **Score = 0.0**: Orthogonal (completely unrelated).
* **Score = -1.0**: Directly opposite meaning.

```mermaid
graph TD
    TextChunk[Text Chunks] --> GeminiEmbed[Google Gemini text-embedding-004]
    GeminiEmbed --> Vectors[768-Dimension Vectors]
    Vectors --> VectorDB[(Vector Store: InMemory / Chroma)]
    Query[User Query] --> QueryEmbed[Query Embedding]
    QueryEmbed --> Similarity[Cosine Similarity Search]
    VectorDB --> Similarity --> TopDocs[Top Relevant Documents]
```

In [1]:
# Part 1 Code: Cosine Similarity from First Principles
import math

def dot_product(v1, v2):
    return sum(x * y for x, y in zip(v1, v2))

def cosine_similarity(v1, v2):
    mag1 = math.sqrt(sum(x * x for x in v1))
    mag2 = math.sqrt(sum(x * x for x in v2))
    if mag1 == 0 or mag2 == 0:
        return 0.0
    return dot_product(v1, v2) / (mag1 * mag2)

# Simulated 3D embeddings:
# v_query: "How to build LangChain pipelines?"
# v_doc1:  "LangChain LCEL tutorial and examples" (High semantic match)
# v_doc2:  "Chocolate cake baking recipe"         (Low semantic match)
v_query = [0.85, 0.90, 0.10]
v_doc1  = [0.82, 0.88, 0.15]
v_doc2  = [0.05, 0.12, 0.95]

score_doc1 = cosine_similarity(v_query, v_doc1)
score_doc2 = cosine_similarity(v_query, v_doc2)

print("=" * 60)
print("📐 MATHEMATICAL SIMILARITY COMPARISON:")
print("=" * 60)
print(f"Cosine Similarity (Query vs Doc 1 - AI Topic):      {score_doc1:.4f}  (High Match!)")
print(f"Cosine Similarity (Query vs Doc 2 - Cooking Topic): {score_doc2:.4f}  (Low Match)")

📐 MATHEMATICAL SIMILARITY COMPARISON:
Cosine Similarity (Query vs Doc 1 - AI Topic):      0.9990  (High Match!)
Cosine Similarity (Query vs Doc 2 - Cooking Topic): 0.2062  (Low Match)


## Part 2: LangChain `Document` Abstraction & Real Google Gemini Embeddings

LangChain provides standard interfaces for representing knowledge chunks:
1. **`Document`**: Encapsulates `page_content` (the text chunk) and `metadata` (source file, topic, date, author).
2. **`GoogleGenerativeAIEmbeddings`**: Uses Google's state-of-the-art `models/text-embedding-004` model.

In [2]:
# Part 2 Code: Real Google Gemini Embeddings & Documents
import os
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

# 1. Create knowledge chunks using genuine LangChain Document objects
docs = [
    Document(
        page_content="LangChain is an open-source framework for building applications with LLMs using LCEL.",
        metadata={"source": "unit3_guide.txt", "topic": "LangChain", "unit": 3}
    ),
    Document(
        page_content="Conversational memory stores multi-turn dialogue history to overcome LLM statelessness.",
        metadata={"source": "unit3_guide.txt", "topic": "Memory", "unit": 3}
    ),
    Document(
        page_content="Output parsers validate and transform free-form model responses into structured Pydantic schemas.",
        metadata={"source": "unit3_guide.txt", "topic": "Output Parsers", "unit": 3}
    ),
    Document(
        page_content="Vector stores index document embeddings to enable fast cosine similarity retrieval in RAG systems.",
        metadata={"source": "unit3_guide.txt", "topic": "Vector Stores", "unit": 3}
    ),
    Document(
        page_content="Preheat the oven to 350 degrees Fahrenheit and line a baking sheet with parchment paper.",
        metadata={"source": "recipes.txt", "topic": "Cooking", "unit": 0}
    )
]

# 2. Initialize Real Google Gemini Embeddings model (text-embedding-004)
gemini_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/text-embedding-004",
    api_key=api_key or "AIzaSy_placeholder_until_env_key_set"
)

try:
    sample_vector = gemini_embeddings.embed_query("What is LangChain?")
    print("=" * 60)
    print("🧬 REAL GOOGLE GEMINI EMBEDDING VECTOR:")
    print("=" * 60)
    print(f"Vector dimensions: {len(sample_vector)} (Google text-embedding-004)")
    print(f"First 5 dimensions: {sample_vector[:5]}")
except Exception as e:
    print("ℹ️ Set your GEMINI_API_KEY in .env to compute live embeddings with Google Gemini.")
    print(f"Notice: {e}")

ℹ️ Set your GEMINI_API_KEY in .env to compute live embeddings with Google Gemini.
Notice: Error embedding content (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}


## Part 3: LangChain `InMemoryVectorStore` & Semantic Search

`InMemoryVectorStore` indexes document vectors and executes nearest-neighbor similarity searches using cosine distance.

In [3]:
# Part 3 Code: Vector Store Indexing and Semantic Search
from langchain_core.vectorstores import InMemoryVectorStore

try:
    # 1. Create and populate Vector Store from documents using Gemini embeddings
    vector_store = InMemoryVectorStore.from_documents(
        documents=docs,
        embedding=gemini_embeddings
    )

    # 2. Perform Similarity Search with Relevance Scores
    query = "How do we store chat history in chatbots?"
    search_results = vector_store.similarity_search_with_score(query, k=2)

    print("=" * 60)
    print(f"🔍 REAL GEMINI SEARCH RESULTS FOR: '{query}'")
    print("=" * 60)

    for rank, (doc, score) in enumerate(search_results, 1):
        print(f"Rank {rank} [Score: {score:.4f}]:")
        print(f"Content: {doc.page_content}")
        print(f"Metadata: {doc.metadata}")
        print("-" * 60)
except Exception as e:
    print("ℹ️ Set your GEMINI_API_KEY in .env to index and search vector store with Gemini.")
    print(f"Notice: {e}")

ℹ️ Set your GEMINI_API_KEY in .env to index and search vector store with Gemini.
Notice: Error embedding content (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}


## Part 4: Connecting Vector Store Retriever to a Gemini RAG Chain

A Vector Store converts into a **Retriever** with `.as_retriever()`.
In LCEL, the retriever automatically extracts relevant context documents and injects them into the final prompt for Gemini!

In [4]:
# Part 4 Code: Complete LCEL RAG Pipeline with Google Gemini
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

def format_docs(retrieved_documents):
    return "\n\n".join(f"[Doc {i+1}]: {d.page_content}" for i, d in enumerate(retrieved_documents))

rag_prompt = ChatPromptTemplate.from_template("""Answer the question based strictly on the provided context.

Context:
{context}

Question: {question}
Answer:""")

rag_gemini_model = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    api_key=api_key or "AIzaSy_placeholder_until_env_key_set",
    temperature=0.3
)

try:
    retriever = vector_store.as_retriever(search_kwargs={"k": 2})

    # Compose LCEL RAG Chain
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | rag_prompt
        | rag_gemini_model
        | StrOutputParser()
    )

    query_question = "Why do we need conversational memory in LLMs?"
    rag_answer = rag_chain.invoke(query_question)

    print("=" * 60)
    print("🎯 REAL GEMINI RETRIEVAL-AUGMENTED GENERATION (RAG) ANSWER:")
    print("=" * 60)
    print(rag_answer)
except Exception as e:
    print("ℹ️ Set your GEMINI_API_KEY in .env to run the live RAG chain with Gemini.")
    print(f"Notice: {e}")

ℹ️ Set your GEMINI_API_KEY in .env to run the live RAG chain with Gemini.
Notice: name 'vector_store' is not defined


### 💡 Theoretical Note: Persistent Vector Databases for Production
In production RAG systems, vectors are stored on disk or in managed cloud vector databases:

```python
# --- ChromaDB (Local Persistent) ---
# pip install chromadb langchain-community
from langchain_community.vectorstores import Chroma
# vector_store = Chroma.from_documents(docs, gemini_embeddings, persist_directory="./chroma_db")

# --- FAISS (Facebook AI Similarity Search - In-Memory High Speed) ---
# pip install faiss-cpu
from langchain_community.vectorstores import FAISS
# vector_store = FAISS.from_documents(docs, gemini_embeddings)
```

## Part 5: Hands-On Student Exercise

**Objective**: Build a "Course Syllabus Search Engine" with Google Gemini:
1. Define a 4-document knowledge base containing descriptions of PGDCA Units 1, 2, 3, and 4.
2. Index them in an `InMemoryVectorStore` using Gemini embeddings.
3. Query the store for "Which unit covers local models and Ollama?" and retrieve the matching unit with its metadata.

In [5]:
# Student Exercise: Course Syllabus Search Engine with Gemini
syllabus_docs = [
    Document(
        page_content="Unit 1 covers Python basics, OpenAI/Gemini SDKs, prompt engineering, and conversational assistants.",
        metadata={"unit": 1, "code": "LLM-P1"}
    ),
    Document(
        page_content="Unit 2 covers parameter tuning, temperature, streaming responses, asyncio concurrency, and cost calculators.",
        metadata={"unit": 2, "code": "LLM-P2"}
    ),
    Document(
        page_content="Unit 3 covers LangChain framework, LCEL chains, memory management, output parsers, and vector stores.",
        metadata={"unit": 3, "code": "LLM-P3"}
    ),
    Document(
        page_content="Unit 4 covers local LLM execution, Ollama setups, LM Studio GGUF benchmarks, and multimodal vision models.",
        metadata={"unit": 4, "code": "LLM-P4"}
    )
]

try:
    syllabus_store = InMemoryVectorStore.from_documents(syllabus_docs, gemini_embeddings)
    student_search_query = "Where can I learn about running local models with Ollama?"
    matched_docs = syllabus_store.similarity_search(student_search_query, k=1)

    print("=" * 60)
    print(f"🎓 SYLLABUS SEARCH QUERY: '{student_search_query}'")
    print("=" * 60)
    top_match = matched_docs[0]
    print(f"Matched Syllabus Unit: Unit {top_match.metadata['unit']} ({top_match.metadata['code']})")
    print(f"Description: {top_match.page_content}")
except Exception as e:
    print("ℹ️ Set your GEMINI_API_KEY in .env to run this search engine live with Gemini.")
    print(f"Notice: {e}")

ℹ️ Set your GEMINI_API_KEY in .env to run this search engine live with Gemini.
Notice: Error embedding content (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}
